# **Indian Premier League**

1. Load and Explore the Data

Import the necessary libraries: pandas and sqlite3.
Connect to the IPL database and load the master table to understand the structure.
Load all the tables and print their column names to identify common columns.

In [2]:
from google.colab import files
uploaded = files.upload()


Saving database.sqlite to database (1).sqlite


In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('database.sqlite')

# watch tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables found:")
print(tables)


Tables found:
               name
0            Player
1        Extra_Runs
2    Batsman_Scored
3     Batting_Style
4     Bowling_Style
5           Country
6            Season
7              City
8           Outcome
9            Win_By
10     Wicket_Taken
11            Venue
12       Extra_Type
13         Out_Type
14    Toss_Decision
15           Umpire
16             Team
17     Ball_by_Ball
18      sysdiagrams
19  sqlite_sequence
20            Match
21            Rolee
22     Player_Match


In [4]:
# show columns of each table
for table in tables['name']:
    print(f"\nColumns in table {table}:")
    df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn)
    print(df.columns.tolist())



Columns in table Player:
['Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling_skill', 'Country_Name']

Columns in table Extra_Runs:
['Match_Id', 'Over_Id', 'Ball_Id', 'Extra_Type_Id', 'Extra_Runs', 'Innings_No']

Columns in table Batsman_Scored:
['Match_Id', 'Over_Id', 'Ball_Id', 'Runs_Scored', 'Innings_No']

Columns in table Batting_Style:
['Batting_Id', 'Batting_hand']

Columns in table Bowling_Style:
['Bowling_Id', 'Bowling_skill']

Columns in table Country:
['Country_Id', 'Country_Name']

Columns in table Season:
['Season_Id', 'Man_of_the_Series', 'Orange_Cap', 'Purple_Cap', 'Season_Year']

Columns in table City:
['City_Id', 'City_Name', 'Country_id']

Columns in table Outcome:
['Outcome_Id', 'Outcome_Type']

Columns in table Win_By:
['Win_Id', 'Win_Type']

Columns in table Wicket_Taken:
['Match_Id', 'Over_Id', 'Ball_Id', 'Player_Out', 'Kind_Out', 'Fielders', 'Innings_No']

Columns in table Venue:
['Venue_Id', 'Venue_Name', 'City_Id']

Columns in table Extra_Type:
['Extra_

2. Query 1: Select All Columns from Player’s Table

Write and execute a SQL query to select all columns from the Player_Match table.

In [30]:
query = '''
SELECT * FROM Ball_by_Ball;
'''
df1 = pd.read_sql_query(query, conn)
df1.head()


,Match_Id,Over_Id,Ball_Id,Innings_No,Team_Batting,Team_Bowling,Striker_Batting_Position,Striker,Non_Striker,Bowler
0,335987,1,1,1,1,2,1,1,2,14
1,335987,1,1,2,2,1,1,6,7,106
2,335987,1,2,1,1,2,2,2,1,14
3,335987,1,2,2,2,1,2,7,6,106
4,335987,1,3,1,1,2,2,2,1,14


3. Query 2: Batsman vs Runs

Write and execute a SQL query to calculate the total runs scored by each batsman.

In [31]:
query = '''
SELECT p.Player_Name, SUM(bs.Runs_Scored) AS Total_Runs
FROM Batsman_Scored bs
JOIN Ball_by_Ball bb
  ON bs.Match_Id = bb.Match_Id
 AND bs.Over_Id = bb.Over_Id
 AND bs.Ball_Id = bb.Ball_Id
JOIN Player p
  ON bb.Striker = p.Player_Id
GROUP BY p.Player_Name
ORDER BY Total_Runs DESC;
'''
df2 = pd.read_sql_query(query, conn)
df2.head()


,Player_Name,Total_Runs
0,V Kohli,8158
1,SK Raina,7588
2,RG Sharma,7263
3,G Gambhir,6793
4,RV Uthappa,6303


4. Query 3: Fifties and Hundreds

Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman.

In [32]:
query = '''
SELECT
    p.Player_Name,
    SUM(CASE WHEN innings_runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END) AS Fifties,
    SUM(CASE WHEN innings_runs >= 100 THEN 1 ELSE 0 END) AS Hundreds
FROM (
    SELECT
        bb.Striker,
        bs.Match_Id,
        SUM(bs.Runs_Scored) AS innings_runs
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball bb
      ON bs.Match_Id = bb.Match_Id
     AND bs.Over_Id = bb.Over_Id
     AND bs.Ball_Id = bb.Ball_Id
    GROUP BY bb.Striker, bs.Match_Id
) AS player_innings
JOIN Player p ON p.Player_Id = player_innings.Striker
GROUP BY p.Player_Name
ORDER BY Hundreds DESC, Fifties DESC;
'''
df3 = pd.read_sql_query(query, conn)
df3.head()


,Player_Name,Fifties,Hundreds
0,V Kohli,29,32
1,DA Warner,21,28
2,G Gambhir,32,24
3,CH Gayle,19,24
4,AM Rahane,18,21


5. Query 4: Best Bowling Figures

Write and execute a SQL query to find the best bowling figures for each bowler.

In [33]:
query = '''
SELECT
    p.Player_Name,
    COUNT(w.Player_Out) AS wickets_taken
FROM Wicket_Taken w
JOIN Ball_by_Ball bb
  ON w.Match_Id = bb.Match_Id
 AND w.Over_Id = bb.Over_Id
 AND w.Ball_Id = bb.Ball_Id
JOIN Player p
  ON bb.Bowler = p.Player_Id
GROUP BY p.Player_Name
ORDER BY wickets_taken DESC;
'''
df4 = pd.read_sql_query(query, conn)
df4.head()


,Player_Name,wickets_taken
0,SL Malinga,292
1,DJ Bravo,236
2,R Vinay Kumar,233
3,Harbhajan Singh,230
4,A Mishra,223


6. Query 5: Comprehensive Career Metrics

Combine all the previous chunks into a single comprehensive query to get detailed career metrics for players.

In [34]:
query = '''
WITH player_innings AS (
    SELECT
        bb.Striker,
        bs.Match_Id,
        SUM(bs.Runs_Scored) AS innings_runs
    FROM Batsman_Scored bs
    JOIN Ball_by_Ball bb
      ON bs.Match_Id = bb.Match_Id
     AND bs.Over_Id = bb.Over_Id
     AND bs.Ball_Id = bb.Ball_Id
    GROUP BY bb.Striker, bs.Match_Id
)

SELECT
    p.Player_Name,
    COALESCE(SUM(CASE WHEN bb.Striker = p.Player_Id THEN bs.Runs_Scored ELSE 0 END), 0) AS total_runs,
    COALESCE(SUM(CASE WHEN pi.innings_runs BETWEEN 50 AND 99 THEN 1 ELSE 0 END), 0) AS fifties,
    COALESCE(SUM(CASE WHEN pi.innings_runs >= 100 THEN 1 ELSE 0 END), 0) AS hundreds,
    COALESCE(COUNT(CASE WHEN bb.Bowler = p.Player_Id AND w.Player_Out IS NOT NULL THEN 1 END), 0) AS wickets_taken
FROM Player p
LEFT JOIN Player_Match pm ON p.Player_Id = pm.Player_Id
LEFT JOIN Ball_by_Ball bb ON pm.Match_Id = bb.Match_Id
LEFT JOIN Batsman_Scored bs ON bb.Match_Id = bs.Match_Id AND bb.Over_Id = bs.Over_Id AND bb.Ball_Id = bs.Ball_Id
LEFT JOIN Wicket_Taken w ON bb.Match_Id = w.Match_Id AND bb.Over_Id = w.Over_Id AND bb.Ball_Id = w.Ball_Id
LEFT JOIN player_innings pi ON pi.Striker = p.Player_Id AND pi.Match_Id = bb.Match_Id
GROUP BY p.Player_Name
ORDER BY total_runs DESC;
'''
df5 = pd.read_sql_query(query, conn)
df5.head()


,Player_Name,total_runs,fifties,hundreds,wickets_taken
0,V Kohli,8158,12677,14749,32
1,SK Raina,7589,20138,8629,133
2,RG Sharma,7264,16626,9219,54
3,G Gambhir,6793,14503,10962,0
4,RV Uthappa,6303,18052,6896,0
